## 1 Типы признаков и шкалы измерения

**Задание 1.**

**1. Типы шкал для каждого признака датасета:**
* `Date` — **Интервальная** (даты имеют равные интервалы — дни, но нулевой точки отсчета в абсолютном смысле нет).
* `Start_Hour`, `End_Hour` — **Интервальная** (время суток циклично, разница между часами имеет смысл).
* `Source` — **Номинальная** (категории "Ветер", "Солнце" без математического порядка).
* `Day_of_Year` — **Интервальная**.
* `Day_Name`, `Month_Name`, `Season` — **Порядковая** (присутствует хронологический порядок: понедельник перед вторником, зима перед весной).
* `Production` — **Шкала отношений / абсолютная** (есть абсолютный ноль — 0 МВт*ч означает полное отсутствие генерации).

**2. Анализ агрегатов для двух признаков с разными шкалами:**
Выберем `Source` (номинальная) и `Production` (шкала отношений).

* **Среднее:**
    * `Source`: Математически некорректно. Pandas выдаст ошибку или бессмысленный результат для строк.
    * `Production`: Корректно. Показывает среднюю генерацию в час.
* **Медиана:**
    * `Source`: Методологически неверно (категории не упорядочены).
    * `Production`: Корректно. Устойчива к выбросам.
* **Мода:**
    * `Source`: Корректно. Покажет наиболее часто встречающийся источник энергии.
    * `Production`: Корректно математически, но часто бесполезно для непрерывных величин.

## 2 Пропуски и их природа (MCAR / MAR / MNAR)

**Задание 2. Диагностика типа пропусков**

Так как в исходном датасете нет пропусков, мы их искусственно сгенерируем для признака `Production`.

**Три альтернативные гипотезы:**
1. **MCAR (Missing Completely At Random):** Пропуски происходят абсолютно случайно (например, случайные сбои датчиков).
2. **MAR (Missing At Random):** Пропуски зависят от других переменных. Например, ночью солнечные панели отключаются, и данных нет именно для `Source == 'Solar'`.
3. **MNAR (Missing Not At Random):** Вероятность пропуска зависит от самого значения. Например, датчик ломается при слишком сильном ветре (очень высокой генерации).

In [14]:
import pandas as pd
import numpy as np
import scipy.stats as stats

df = pd.read_csv('../data/Energy Production Dataset.csv')

df_miss = df.copy()
df_miss['Production'] = df_miss['Production'].astype(float)

# генерируем пропуски (MAR)
np.random.seed(42)
mask_mar = (df_miss['Source'] == 'Solar') & (np.random.rand(len(df_miss)) < 0.2)
df_miss.loc[mask_mar, 'Production'] = np.nan

df_miss['Production_is_missing'] = df_miss['Production'].isnull().astype(int)

#  считаем таблицу
contingency_table = pd.crosstab(df_miss['Source'], df_miss['Production_is_missing'])
print("Таблица сопряженности:")
print(contingency_table)

# тест
chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
print(f"\nХи-квадрат p: {p:.4e}")

if p < 0.05:
    print("Вывод: p < 0.05. Отвергаем MCAR. Пропуски зависят от Source (это MAR).")
else:
    print("Вывод: p > 0.05. Не удалось доказать зависимость (MCAR).")

Таблица сопряженности:
Production_is_missing      0     1
Source                            
Mixed                      2     0
Solar                   7531  1847
Wind                   42484     0

Хи-квадрат p: 0.0000e+00
Вывод: p < 0.05. Отвергаем MCAR. Пропуски зависят от Source (это MAR).


### **Задание 3. Индикатор пропуска как источник информации**

**1. Когда сам факт пропуска важен?**
Пропуск несет информацию, когда он случился не случайно, а из-за особенностей процесса. 
* *Пример:* Если ветряк на ремонте, данных о нем не будет. Но этот пропуск — полезный сигнал: он сам по себе сообщает, что оборудование сейчас отключено.

**2. Почему важность индикатора пропуска — это плохой знак?**
Если мы заполнили пустые места (например, вставили среднее значение), а модель при обучении начала сильно опираться на значок пропуска (**индикатор 0/1**), значит:
* Наша «заплатка» получилась неудачной и слишком выделяется.
* Модель поняла, что вставленные данные не настоящие, и использует этот значок, чтобы отличить их от реальности. Это сигнал, что данные подготовлены некорректно.

**3. Примеры:**
* **Полезен:** Датчик солнечного света. Ночью он не работает и выдает пропуски. Такой пропуск полезен, так как он помогает четко отличить ночь от дня.
* **Бесполезен:** Случайный сбой в передаче данных. Информация просто потерялась из-за помех. Такой пропуск не несет смысла и не помогает модели.
* **Опасен:** Если датчик ломается каждый раз прямо перед перегревом оборудования. Модель увидит пропуск и «предскажет» аварию слишком легко. В реальности мы так сделать не сможем — это подсказка, которой не будет в рабочих условиях (**утечка данных**).

## 3 Выбросы

**Задание 4. Выброс - это ошибка или сигнал?**

* **Числовой признак:** `Production`.
* **Причина:** Экстремальные погодные условия (шторм), дающие пиковую нагрузку.
* **Что хуже:** Удалить этот выброс — хуже всего. Мы потеряем информацию о пиковых нагрузках, важных для стабильности сети.

## 4 Статистики

### **Задание 5. Преобразование или винзоризация?**

Для числового признака с сильной асимметрией (как наш `Production`):

**1. В каких случаях предпочтительнее:**
* **Логарифмирование:** Когда данные различаются в десятки и сотни раз (например, от 10 до 10 000). Оно «сжимает» огромные значения сильнее всего и делает распределение похожим на колокол.
* **Корень (квадратный):** Когда асимметрия есть, но она не такая экстремальная. Корень действует мягче логарифма. Его часто используют для данных, где много нулевых или очень маленьких значений, так как логарифм от нуля взять нельзя.
* **Винзоризация:** Когда мы считаем, что крайние значения (хвосты) — это аномалии или ошибки. Мы не меняем форму всех данных, а просто «подрезаем» самые высокие и низкие значения до определенного уровня.

**2. Как подходы влияют на:**

* **Интерпретируемость:**
    * **Винзоризация** — лучшая (единицы измерения остаются прежними: МВт*ч).
    * **Корень** — средняя (сложнее объяснить, что такое «корень из мегаватта»).
    * **Логарифм** — худшая (приходится говорить об изменениях в процентах или разах).

* **Линейные модели:**
    * **Логарифм и корень** помогают модели лучше работать, если связь между признаками не прямая, а кривая. Они также уменьшают влияние разброса данных.
    * **Винзоризация** просто не дает модели «сходить с ума» из-за гигантских выбросов, которые могут сильно исказить предсказания.

* **Визуализацию распределений:**
    * **Логарифм и корень** превращают «длинный хвост» в компактный график, похожий на нормальное распределение.
    * **Винзоризация** оставляет график почти без изменений, но на самых краях появляются неестественно высокие «столбики», где собраны все обрезанные значения.

In [12]:
print(f"Арифметическое среднее: {df['Production'].mean():.2f}")
print(f"Медиана: {df['Production'].median():.2f}")
print(f"Геометрическое среднее: {stats.gmean(df['Production']):.2f}")

Арифметическое среднее: 6215.07
Медиана: 5372.00
Геометрическое среднее: 4926.75


### **Задание 6. Средние, которые вводят в заблуждение**

Для анализа выбран признак `Production` (объем генерации электроэнергии).

**1. Сравнение показателей:**
* **Арифметическое среднее: 6215.07**
* **Медиана: 5372.00**
* **Геометрическое среднее: 4926.75**

**2. Почему они различаются:**
* **Арифметическое среднее** самое высокое, потому что оно сильно чувствительно к «хвостам» распределения. В нашем датасете есть редкие пики очень высокой генерации (выбросы), которые «тянут» среднее значение вверх.
* **Медиана** находится посередине. Она просто разделяет данные пополам и игнорирует, насколько велики крайние значения. Поэтому она меньше арифметического среднего.
* **Геометрическое среднее** самое низкое. Оно математически всегда меньше или равно арифметическому. Оно сильнее штрафует большие отклонения и «прижимается» к меньшим значениям.

**3. Какое значение лучше отражает «типичное» и почему:**
Лучше всего типичное значение отражает **Медиана (5372.00)**. 
* *Почему:* Она устойчива к аномалиям. Если в какой-то день случился мощный шторм и генерация подскочила до максимума, арифметическое среднее сразу вырастет и создаст ложное впечатление, что мы всегда производим много энергии. Медиана же останется стабильной и покажет реальную картину, которую мы видим в обычный, «типичный» час работы.

## 5 Визуализация

### **Задание 7. Неправильная диаграмма**

1. **Выбранная визуализация:** Boxplot (ящик с усами) для признака `Production`, который мы строили в первой работе. Он отлично показывает распределение и выбросы.
2. **Худший тип диаграммы:** Круговая диаграмма (Pie Chart) для этого же признака, если сгруппировать его по месяцам.
   * **Какую информацию она исказит:** Круговая диаграмма плохо показывает разницу между секторами, если они близки по размеру. Разницу в 5–10% между месяцами на глаз практически не заметить.
   * **Какой неверный вывод сделает зритель:** Зритель решит, что генерация стабильна круглый год, и пропустит важные сезонные просадки или пики, которые были бы сразу видны на столбчатой диаграмме или графике.

---

### **Задание 8. Одна и та же информация — разные графики**

Мы сравнили `Start_Hour` (время суток) и `Production` (объем генерации).

1. **Линейный график (Line plot):**
   * **Что подчеркивает:** Общий **тренд**. Мы сразу видим «горку» или спады: когда в среднем энергия вырабатывается активнее (например, днем).
   * **Гипотеза:** «В дневные часы генерация в среднем выше».

2. **Ящики с усами (Boxplot):**
   * **Что подчеркивает:** **Разброс (дисперсию)** и выбросы. Мы видим, насколько сильно меняется генерация внутри одного часа (стабильна она или скачет).
   * **Гипотеза:** «Ночью генерация не только ниже, но и нестабильнее (сильнее разброс), чем днем». 

**Вывод:** Линейный график идеален для поиска среднего поведения, а Boxplot незаменим для оценки рисков и стабильности системы.